# Response Time Analysis

This notebook aims to explore the distributions and tendencies of response times within the SPD Calls data and the relationships between response time and other variables. The majority of the analysis uses median response times to capture the typical response time and uses geographical location and call type as explanatory variables. 

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from spd_snapshot import load_spd_call_snapshot
from spd_eda import summarize_spd_calls

import warnings
warnings.filterwarnings("ignore")

EVENT_ID_COLUMN = "cad_event_number"
ROW_ID_COLUMN = "call_sign_dispatch_id"

QUEUE_TIME_COLUMN = "cad_event_original_time_queued"
ARRIVAL_TIME_COLUMN = "cad_event_arrived_time"

PLOTLY_TEMPLATE = "plotly_dark"
PLOT_BG = "#545455"
PAPER_BG = "#111111"

df, metadata = load_spd_call_snapshot(
    PROJECT_ROOT / "data" / "processed"
)

display(df.head())

summary = summarize_spd_calls(df)
print(summary)

In [ ]:
response_df = df.copy()

response_df[QUEUE_TIME_COLUMN] = pd.to_datetime(
    response_df[QUEUE_TIME_COLUMN],
    errors="coerce"
)

response_df[ARRIVAL_TIME_COLUMN] = pd.to_datetime(
    response_df[ARRIVAL_TIME_COLUMN],
    errors="coerce"
)

text_cols = [
    "event_group",
    "call_type",
    "priority",
    "initial_call_type",
    "final_call_type",
    "cad_event_response_category",
    "dispatch_neighborhood",
    "dispatch_precinct",
    "dispatch_sector",
    "dispatch_beat",
]

for col in text_cols:
    if col in response_df.columns:
        response_df[col] = (
            response_df[col]
            .astype("string")
            .str.strip()
            .str.lower()
        )

event_response = (
    response_df
    .dropna(subset=[EVENT_ID_COLUMN])
    .sort_values(QUEUE_TIME_COLUMN)
    .groupby(EVENT_ID_COLUMN, as_index=False)
    .agg(
        queued_time=(QUEUE_TIME_COLUMN, "min"),
        first_arrival_time=(ARRIVAL_TIME_COLUMN, "min"),
        event_group=("event_group", "first"),
        call_type=("call_type", "first"),
        priority=("priority", "first"),
        initial_call_type=("initial_call_type", "first"),
        final_call_type=("final_call_type", "first"),
        response_category=("cad_event_response_category", "first"),
        dispatch_neighborhood=("dispatch_neighborhood", "first"),
        dispatch_precinct=("dispatch_precinct", "first"),
        dispatch_sector=("dispatch_sector", "first"),
        dispatch_beat=("dispatch_beat", "first"),
        dispatch_records=(ROW_ID_COLUMN, "nunique"),
    )
)

event_response["response_time_minutes"] = (
    event_response["first_arrival_time"]
    - event_response["queued_time"]
).dt.total_seconds() / 60

event_response["date"] = event_response["queued_time"].dt.date
event_response["month"] = event_response["queued_time"].dt.to_period("M").dt.to_timestamp()
event_response["week"] = event_response["queued_time"].dt.to_period("W").apply(lambda x: x.start_time)
event_response["hour"] = event_response["queued_time"].dt.hour
event_response["day_of_week"] = event_response["queued_time"].dt.day_name()
event_response["day_of_week_num"] = event_response["queued_time"].dt.dayofweek

event_response.head()

Below we see a basic summary of how many observations are available for analysis along with some other metrics.

In [ ]:
response_qa_summary = pd.DataFrame({
    "Metric": [
        "Unique CAD events",
        "Events with queued time",
        "Events with arrival time",
        "Events with both queued and arrival time",
        "Events with nonnegative response time",
        "Events with response time over 12 hours",
        "Events with response time over 24 hours",
    ],
    "Value": [
        event_response[EVENT_ID_COLUMN].nunique(),
        event_response["queued_time"].notna().sum(),
        event_response["first_arrival_time"].notna().sum(),
        event_response[["queued_time", "first_arrival_time"]].notna().all(axis=1).sum(),
        (event_response["response_time_minutes"] >= 0).sum(),
        (event_response["response_time_minutes"] > 12 * 60).sum(),
        (event_response["response_time_minutes"] > 24 * 60).sum(),
    ],
})

response_qa_summary["Percentage"] = (
    response_qa_summary["Value"]
    / event_response[EVENT_ID_COLUMN].nunique()
    * 100
)


# Formatting for readability (nightmare)
response_qa_summary['Percentage'] = response_qa_summary['Percentage'].map(
    lambda x: f"{float(x):.1f}%" if pd.notnull(x) and str(x).strip() != "" else ""
)
response_qa_summary['Value'] = response_qa_summary['Value'].apply(lambda x: f"{x:,.1f}" if pd.notnull(x) else "")

response_qa_summary_print = response_qa_summary.set_index('Metric').T
response_qa_summary_print.index.name = None
response_qa_summary_print = response_qa_summary_print.style.set_properties(**{
    'text-align': 'center'
}).set_table_styles([{
    'selector': 'th',
    'props': [('text-align', 'center')]
}])
response_qa_summary_print

#response_qa_summary

In [ ]:
response_analysis = event_response[
    event_response["response_time_minutes"].notna()
    & (event_response["response_time_minutes"] > 0)
    & (event_response["response_time_minutes"] <= 24 * 60)
].copy()

print(f"Events available for response-time analysis: {len(response_analysis):,}")
print(f"Share of all CAD events: {len(response_analysis) / len(event_response) * 100:.2f}%")

response_analysis["response_time_minutes"].describe()
print(f"Start date: {response_analysis["queued_time"].min()} End date:{response_analysis["queued_time"].max()}")

## Typical Response Times, Response Time by Crime Type 

Below we see a histogram of all available observations' response times. Note the extreme right skew, response times cannot be negative of course and most are between 0-25 minutes. There are some response times that are so incredibly long that they skew the data substantially, ultra-long response times are something that need to be examined so we can learn how to avoid them.

In [ ]:
median = response_analysis['response_time_minutes'].median()
fig = px.histogram(
    response_analysis, 
    x="response_time_minutes",
    range_x=[0, 600], 
    range_y=[0, 120000], 
    nbins=30, 
    title="Response Time (min) Distribution")
fig.add_vline(x=median, line_dash="dash", line_color="red", annotation_text=f"Median: {median:.2f}")
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
)
fig.show()

In [ ]:
def summarize_response_time(data, group_cols):
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    summary = (
        data
        .dropna(subset=group_cols + ["response_time_minutes"])
        .groupby(group_cols, as_index=False)
        .agg(
            unique_call_events=(EVENT_ID_COLUMN, "nunique"),
            mean_response_minutes=("response_time_minutes", "mean"),
            median_response_minutes=("response_time_minutes", "median"),
            p25_response_minutes=("response_time_minutes", lambda x: x.quantile(0.25)),
            p75_response_minutes=("response_time_minutes", lambda x: x.quantile(0.75)),
            p90_response_minutes=("response_time_minutes", lambda x: x.quantile(0.90)),
        )
    )

    return summary

Below we show the median response times for each of the priority levels, please recall that a priority of 1 means the highest priority. As expected the higher priority calls typically have the lowest response time with a median of 7.0 minutes, followed by priority 2 calls with a median response time of 28.7 minutes and priority 3 calls with a median response time of 74.5 minutes. The drop-off between 1 and 2 makes sense, but it is far from ideal that the typical wait time for an SPD response to an incident described as "not presenting direct threat to life but urgent and could easily escalate" is 28.7 minutes. 

In [ ]:
priority_response_summary = summarize_response_time(
    response_analysis,
    "priority"
)

priority_response_summary = (
    priority_response_summary
    .sort_values("priority")
    .reset_index(drop=True)
)

priority_response_summary
fig = px.bar(
    priority_response_summary,
    x="priority",
    y="median_response_minutes",
    error_y=priority_response_summary["p75_response_minutes"] - priority_response_summary["median_response_minutes"],
    title="Median SPD Response Time by Priority",
    labels={
        "priority": "Priority",
        "median_response_minutes": "Median response time (minutes)",
    },
    text="median_response_minutes",
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    yaxis_title="Median response time (minutes)",
    xaxis_title="Priority",
)

fig.show()

Below we examine the median response time among each of the event importance bins we have defined


The bins are as follows:


`violent_or_person_crime_groups`<br>

Includes:<br>
> "assaults", "domestic disturbance/violence", "homicide", "kidnap", "rape", "robbery", "sex offenses (non-rape)", "human trafficking", "arson, bombs, explosion", "shots",
> "vice" (includes human trafficking and prostitution), "child (abandoned, abused, or neglected)", "weapon, person with", "bias" (includes hate crimes)<br>


`drug_related_groups` 

Includes:<br>
> "narcotics", "detox", "crisis complaint" (crisis complaint can be drug related crisis), "intoxication & liquor violations"<br>


`property_or_nonviolent_groups`<br>

Includes:<br>
> "burglary", "theft", "prowler", "fraud", "trespass", "property destruction (damage)", "premise checks", "crisis complaint",
>"automobiles" (theft of a car), "property", "fraud & forgery"<br>


`lower_public_safety_groups`<br>

Includes:<br>
> "traffic", "disturbance", "misc. misdemeanors & violations", "warrant services/order", "person down/injury", "suspicious circumstances", 
> "casualty (non-traffic.non-criminal, including man down, injured & sick person, doa)", "mischief & nuisance", "hazards", "person missing/found", 
> "threats/harassment (no bias)"<br>


`unknown / unclassified`<br>

Includes:<br>
> "alarms, false", "assist the officer", "assist other agency", "info & radio broadcast", "unkown-ani/ali", "assigned duty", "assist public", "follow-ups",
> "harbor (water)", "animal complaint", "administrative", "other", "public gatherings", "custodial interference", "escape/eluding police", "amber alerts", 
> "directed patrol", "predictive policing", "swatting"<br>


In the past year's data events classified as violent/person typically have the fastest responses with a median of 17.3 minutes, followed by other / unclassified crime with a median of 18.5 minutes, then property/nonviolent with a median of 28.4 minutes, then drug-related with a median of 30.7 minutes, then the slowest responses are those belonging to the lower public-safety urgency category with a median of 34.6 minutes. The drug-related response time is far from ideal but typically violent crimes are responded to the fastest

In [ ]:
violent_or_person_crime_groups = [
    "assaults",
    "domestic disturbance/violence",
    "homicide",
    "kidnap",
    "rape",
    "robbery",
    "sex offenses (non-rape)",
    "human trafficking",
    "arson, bombs, explosion",
    "shots",
    "vice",
    "child (abandoned, abused, or neglected)",
    "weapon, person with",
    "bias"
]

drug_related_groups = [
    "narcotics",
    "detox",
    "crisis complaint",
]

property_or_nonviolent_groups = [
    "burglary",
    "theft",
    "prowler",
    "fraud",
    "trespass",
    "property destruction (damage)",
    "premise checks",
    "crisis complaint",
    "intoxication & liquor violations",
    "automobiles",
    "property",
    "fraud & forgery",
	
]

lower_public_safety_groups = [
    "traffic",
    "disturbance",
    "misc. misdemeanors & violations",
    "warrant services/order",
    "person down/injury",
    "suspicious circumstances",
    "casualty (non-traffic. non-criminal, including man down, injured & sick person, doa)",
    "mischief & nuisance",
    "hazards",
    "person missing/found",
    "threats/harassment (no bias)",
]


def assign_event_importance_bin(event_group):
    if pd.isna(event_group):
        return "unknown / unclassified"

    event_group = str(event_group).strip().lower()

    if event_group in violent_or_person_crime_groups:
        return "violent/person crime"

    if event_group in drug_related_groups:
        return "drug-related"

    if event_group in property_or_nonviolent_groups:
        return "property/nonviolent"

    if event_group in lower_public_safety_groups:
        return "lower public-safety urgency"

    return "other / unclassified"


response_analysis["event_importance_bin"] = (
    response_analysis["event_group"]
    .apply(assign_event_importance_bin)
)

response_analysis["event_importance_bin"].value_counts()

importance_response_summary = summarize_response_time(
    response_analysis,
    "event_importance_bin"
)

importance_response_summary = (
    importance_response_summary
    .sort_values("median_response_minutes")
    .reset_index(drop=True)
)

importance_response_summary
fig = px.bar(
    importance_response_summary,
    x="median_response_minutes",
    y="event_importance_bin",
    orientation="h",
    title="Median SPD Response Time by Event Importance Bin",
    labels={
        "median_response_minutes": "Median response time (minutes)",
        "event_importance_bin": "Event importance bin",
    },
    text="median_response_minutes",
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    xaxis_title="Median response time (minutes)",
    yaxis_title="Event importance bin",
)

fig.show()

Below we examine the response time distribution for violent crimes, drug-related crimes, and property/nonviolent crimes. It is notable that even with the large volume of violent crimes the median response time within that group is still the lowest. Drug related crimes have the lowest volume out of the three plotted but still has the largest median 

In [ ]:
response_analysis["response_time_minutes"] = pd.to_numeric(
    response_analysis["response_time_minutes"], errors="coerce"
)

target_bins = ["violent/person crime", "drug-related", "property/nonviolent"]
hist_df = response_analysis[
    (response_analysis["event_importance_bin"].isin(target_bins))
    & (response_analysis["response_time_minutes"] > 0)
].copy()

violent_med = importance_response_summary.loc[
    importance_response_summary["event_importance_bin"] == "violent/person crime",
    "median_response_minutes",
].values[0]

drug_med = importance_response_summary.loc[
    importance_response_summary["event_importance_bin"] == "drug-related",
    "median_response_minutes",
].values[0]

property_med = importance_response_summary.loc[
    importance_response_summary["event_importance_bin"] == "property/nonviolent",
    "median_response_minutes",
].values[0]

fig = px.histogram(
    hist_df,
    x="response_time_minutes",
    facet_row="event_importance_bin",
    color="event_importance_bin",
    nbins=100,  
    range_x=[0, 240], 
    range_y=[0, 14500],
    title="Response Time Distribution: Violent vs. Drug-Related Crimes",
    labels={"response_time_minutes": "Response Time (minutes)"},
    category_orders={"event_importance_bin": target_bins},
    facet_row_spacing=0.12,
)

fig.add_vline(
    x=violent_med,
    line_dash="dash",
    line_color="red",
    line_width=2,
    row=3,
    col=1,
    annotation_text=f"Median: {violent_med:.2f}m",
    annotation_position="top right",
)

fig.add_vline(
    x=drug_med,
    line_dash="dash",
    line_color="darkblue",
    line_width=2,
    row=2,
    col=1,
    annotation_text=f"Median: {drug_med:.2f}m",
    annotation_position="top right",
)

fig.add_vline(
    x=property_med,
    line_dash="dash",
    line_color="green",
    line_width=2,
    row=1,
    col=1,
    annotation_text=f"Median: {property_med:.2f}m",
    annotation_position="top right",
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    showlegend=False,
)

fig.for_each_annotation(lambda a: a.update(
    text=a.text.split("=")[-1].title(),
    textangle=0,            
    x=0.5,                  
    xanchor="center",  
    yanchor="bottom",
) if "=" in a.text else None)

fig.update_yaxes(matches=None)
fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))
fig.show()


The typical priority assigned for each group gives us more insight on the response times. Below we can see that the most frequent priorities assigned for each group are:

Violent/Person Crime: 2


Drug-Related: 3


Property/Nonviolent: 2

Property related crimes do have a second peak at the priority level of 5 (lower priority) which likely contributes to the longer response times, but the most revealing observation is the mode of the drug-related crimes being priority 3. Drug related crimes are frequently labeled as a lower priority than violent crimes, and are often labeled as a lower priority than property/nonviolent crimes.

In [ ]:
target_bins = ["violent/person crime", "drug-related", "property/nonviolent"]
facet_df = response_analysis[response_analysis["event_importance_bin"].isin(target_bins)].copy()

facet_df["priority"] = pd.to_numeric(facet_df["priority"], errors="coerce")

fig = px.histogram(
    facet_df,
    x="priority",
    facet_col="event_importance_bin",
    color="event_importance_bin",
    nbins=10,
    title="Priority Distribution by Event Importance Bin",
    labels={"priority": "Priority Code"},
    category_orders={"event_importance_bin": target_bins},
    facet_col_spacing=0.05,
)

fig.update_yaxes(matches=None)
fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].title()))

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    showlegend=False,
    xaxis_title="Priority Code",
)

fig.show()


## Geographical Analysis of Response Times

Below we see the median repsonse time for calls of any group for each of the 15 slowest neighborhoods by response time. None of these neighborhoods seem to have a particularly bad reputation crime-wise, so it is interesting that the typical call response among all groups for these neighborhoods is low.

In [ ]:
neighborhood_response_summary = summarize_response_time(
    response_analysis,
    "dispatch_neighborhood"
)

neighborhood_response_summary = neighborhood_response_summary[
    ~neighborhood_response_summary["dispatch_neighborhood"].isin(["unknown", "-", "", "nan"])
].copy()

neighborhood_response_summary = (
    neighborhood_response_summary
    .sort_values("median_response_minutes", ascending=False)
    .reset_index(drop=True)
)

neighborhood_response_summary.head(20)
MIN_NEIGHBORHOOD_EVENTS = 250

neighborhood_response_reliable = neighborhood_response_summary[
    neighborhood_response_summary["unique_call_events"] >= MIN_NEIGHBORHOOD_EVENTS
].copy()

print(f"Neighborhoods with at least {MIN_NEIGHBORHOOD_EVENTS:,} events: {len(neighborhood_response_reliable)}")
slowest_neighborhoods = (
    neighborhood_response_reliable
    .sort_values("median_response_minutes", ascending=False)
    .head(15)
    .sort_values("median_response_minutes")
)

fig = px.bar(
    slowest_neighborhoods,
    x="median_response_minutes",
    y="dispatch_neighborhood",
    orientation="h",
    title=f"Slowest SPD Response Times by Neighborhood, Minimum {MIN_NEIGHBORHOOD_EVENTS:,} Events (all event groups)",
    labels={
        "median_response_minutes": "Median response time minutes",
        "dispatch_neighborhood": "Dispatch neighborhood",
    },
    text="median_response_minutes",
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
)

fig.show()

Below we see the same figure as above but for the neighborhoods with the fastest response times. Most of these neighborhoods do have somewhat of a reputation for a high volume of crime. It seems to be that the more prevalent crime is in the area the faster the response time when analyzing all groups.

In [ ]:
fastest_neighborhoods = (
    neighborhood_response_reliable
    .sort_values("median_response_minutes", ascending=True)
    .head(15)
    .sort_values("median_response_minutes", ascending=False)
)

fig = px.bar(
    fastest_neighborhoods,
    x="median_response_minutes",
    y="dispatch_neighborhood",
    orientation="h",
    title=f"Fastest SPD Response Times by Neighborhood, Minimum {MIN_NEIGHBORHOOD_EVENTS:,} Events (all event groups)",
    labels={
        "median_response_minutes": "Median response time minutes",
        "dispatch_neighborhood": "Dispatch neighborhood",
    },
    text="median_response_minutes",
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
)

fig.show()

In [ ]:
def plot_slowest_neighborhoods_for_importance_bin(
    response_analysis,
    target_importance_bin,
    min_neighborhood_events=100,
    top_n=15,
):
    bin_response = response_analysis[
        response_analysis["event_importance_bin"] == target_importance_bin
    ].copy()

    #print(f"Target bin: {target_importance_bin}")
    #print(f"Events in bin: {len(bin_response):,}")
    #print()
    #print("Event groups in this bin:")
    #display(bin_response["event_group"].value_counts(dropna=False).head(20))

    bin_neighborhood_response_summary = summarize_response_time(
        bin_response,
        "dispatch_neighborhood"
    )

    bin_neighborhood_response_summary = bin_neighborhood_response_summary[
        ~bin_neighborhood_response_summary["dispatch_neighborhood"].isin(
            ["unknown", "-", "", "nan"]
        )
    ].copy()

    bin_neighborhood_response_summary = (
        bin_neighborhood_response_summary
        .sort_values("median_response_minutes", ascending=False)
        .reset_index(drop=True)
    )

    bin_neighborhood_response_reliable = bin_neighborhood_response_summary[
        bin_neighborhood_response_summary["unique_call_events"] >= min_neighborhood_events
    ].copy()

    #print(
    #    f"Neighborhoods with at least {min_neighborhood_events:,} "
    #    f"{target_importance_bin} events: {len(bin_neighborhood_response_reliable)}"
    #)

    slowest_neighborhoods = (
        bin_neighborhood_response_reliable
        .sort_values("median_response_minutes", ascending=False)
        .head(top_n)
        .sort_values("median_response_minutes")
    )

    fig = px.bar(
        slowest_neighborhoods,
        x="median_response_minutes",
        y="dispatch_neighborhood",
        orientation="h",
        title=(
            f"Slowest SPD Response Times by Neighborhood, "
            f"Minimum {min_neighborhood_events:,} Events ({target_importance_bin})"
        ),
        labels={
            "median_response_minutes": "Median response time minutes",
            "dispatch_neighborhood": "Dispatch neighborhood",
        },
        text="median_response_minutes",
        hover_data={
            "unique_call_events": ":,",
            "mean_response_minutes": ":.1f",
            "p75_response_minutes": ":.1f",
            "p90_response_minutes": ":.1f",
        },
    )

    fig.update_traces(
        texttemplate="%{text:.1f}",
        textposition="outside",
        cliponaxis=False,
    )

    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        plot_bgcolor=PLOT_BG,
        paper_bgcolor=PAPER_BG,
        xaxis_title="Median response time minutes",
        yaxis_title="Dispatch neighborhood",
    )

    fig.show()

    return bin_neighborhood_response_summary, bin_neighborhood_response_reliable

def plot_fastest_neighborhoods_for_importance_bin(
    response_analysis,
    target_importance_bin,
    min_neighborhood_events=100,
    top_n=15,
):
    bin_response = response_analysis[
        response_analysis["event_importance_bin"] == target_importance_bin
    ].copy()

    #print(f"Target bin: {target_importance_bin}")
    #print(f"Events in bin: {len(bin_response):,}")
    #print()
    #print("Event groups in this bin:")
    #display(bin_response["event_group"].value_counts(dropna=False).head(20))

    bin_neighborhood_response_summary = summarize_response_time(
        bin_response,
        "dispatch_neighborhood"
    )

    bin_neighborhood_response_summary = bin_neighborhood_response_summary[
        ~bin_neighborhood_response_summary["dispatch_neighborhood"].isin(
            ["unknown", "-", "", "nan"]
        )
    ].copy()

    bin_neighborhood_response_summary = (
        bin_neighborhood_response_summary
        .sort_values("median_response_minutes", ascending=True)
        .reset_index(drop=True)
    )

    bin_neighborhood_response_reliable = bin_neighborhood_response_summary[
        bin_neighborhood_response_summary["unique_call_events"] >= min_neighborhood_events
    ].copy()

    #print(
    #    f"Neighborhoods with at least {min_neighborhood_events:,} "
    #    f"{target_importance_bin} events: {len(bin_neighborhood_response_reliable)}"
    #)

    fastest_neighborhoods = (
        bin_neighborhood_response_reliable
        .sort_values("median_response_minutes", ascending=True)
        .head(top_n)
        .sort_values("median_response_minutes", ascending=False)
    )

    fig = px.bar(
        fastest_neighborhoods,
        x="median_response_minutes",
        y="dispatch_neighborhood",
        orientation="h",
        title=(
            f"Fastest SPD Response Times by Neighborhood, "
            f"Minimum {min_neighborhood_events:,} Events ({target_importance_bin})"
        ),
        labels={
            "median_response_minutes": "Median response time minutes",
            "dispatch_neighborhood": "Dispatch neighborhood",
        },
        text="median_response_minutes",
        hover_data={
            "unique_call_events": ":,",
            "mean_response_minutes": ":.1f",
            "p75_response_minutes": ":.1f",
            "p90_response_minutes": ":.1f",
        },
    )

    fig.update_traces(
        texttemplate="%{text:.1f}",
        textposition="outside",
        cliponaxis=False,
    )

    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        plot_bgcolor=PLOT_BG,
        paper_bgcolor=PAPER_BG,
        xaxis_title="Median response time minutes",
        yaxis_title="Dispatch neighborhood",
    )

    fig.show()

    return bin_neighborhood_response_summary, bin_neighborhood_response_reliable

### Slowest Response times by Crime Bin

Below we see bar charts for each of the following crime bins (defined previously in the notebook). Within the violent event bar chart we see the lowest median response time out of any bin, confirming that violent crimes are typically responded to the fastest; we can also see that Roosevelt/Ravenna is the neighborhood with the slowest median response time (40.1 minutes). Within the drug-related event bar chary we can see that the median response times tend to be much larger, and Roosevelt/Ravenna has the slowest median resposne time once again (81.9 minutes). And finally Property/Nonviolent crimes show middling response times, and the slowest median response time belongs to Central Area/Squire Park.

In [ ]:
violent_neighborhood_summary, violent_neighborhood_reliable = plot_slowest_neighborhoods_for_importance_bin(
    response_analysis=response_analysis,
    target_importance_bin="violent/person crime",
    min_neighborhood_events=250,
    top_n=15,
)

In [ ]:
drug_neighborhood_summary, drug_neighborhood_reliable = plot_slowest_neighborhoods_for_importance_bin(
    response_analysis=response_analysis,
    target_importance_bin="drug-related",
    min_neighborhood_events=250,
    top_n=15,
)

In [ ]:
property_neighborhood_summary, property_neighborhood_reliable = plot_slowest_neighborhoods_for_importance_bin(
    response_analysis=response_analysis,
    target_importance_bin="property/nonviolent",
    min_neighborhood_events=250,
    top_n=15,
)

### Fastest Response times by Crime Bin

Below we see the fastest response times, in which we see further evidence that the higher the prevalence of crime the lower the typical response times. Capitol Hill has the fastest median response times for both violent and drug-related crimes and Judkins Park/North Beacon Hill has the fastest median response for property related crimes.

In [ ]:
violent_fastest_summary, violent_fastest_reliable = plot_fastest_neighborhoods_for_importance_bin(
    response_analysis=response_analysis,
    target_importance_bin="violent/person crime",
    min_neighborhood_events=250,
    top_n=15,
)

In [ ]:
drug_fastest_summary, drug_fastest_reliable = plot_fastest_neighborhoods_for_importance_bin(
    response_analysis=response_analysis,
    target_importance_bin="drug-related",
    min_neighborhood_events=250,
    top_n=15,
)

In [ ]:
property_fastest_summary, property_fastest_reliable = plot_fastest_neighborhoods_for_importance_bin(
    response_analysis=response_analysis,
    target_importance_bin="property/nonviolent",
    min_neighborhood_events=250,
    top_n=15,
)

The cell below does provide insightful info for the median response time and displays it in an intuitive manner, it would likely be something hidden away in a submenu however because it does not exactly tie into the chloropleth of call volume with points for recent crimes. We could maybe shoehorn it in there, give the regions hover data that displays median response time, share of important events (could be broken up into important events, medium importance events, low priority events), perhaps count of events, and the ranking of the neighborhood in terms of share or volume of important events. We could include the response time in the points as well. 

In [ ]:
import geopandas as gpd
import json
import pandas as pd
import plotly.express as px

# ------------------------------------------------------------
# Response-time choropleth using official SPD MCPP boundaries
# Works from a non-geography notebook
# Requires:
#   response_analysis
#   event_mcpp_lookup.parquet saved from 03_geography_and_mapping
# ------------------------------------------------------------

GEO_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "geography"

mcpp_event_lookup_path = GEO_PROCESSED_DIR / "event_mcpp_lookup.parquet"
processed_mcpp_geojson_path = GEO_PROCESSED_DIR / "spd_mcpp_boundaries.geojson"

# ------------------------------------------------------------
# Load saved MCPP event lookup
# ------------------------------------------------------------

event_mcpp_lookup = pd.read_parquet(mcpp_event_lookup_path)

event_mcpp_lookup["mcpp_neighborhood"] = (
    event_mcpp_lookup["mcpp_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

event_mcpp_lookup["mcpp_precinct"] = (
    event_mcpp_lookup["mcpp_precinct"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# ------------------------------------------------------------
# Load official MCPP boundary GeoJSON
# ------------------------------------------------------------

mcpp_choropleth_gdf = gpd.read_file(processed_mcpp_geojson_path)

if mcpp_choropleth_gdf.crs is not None:
    mcpp_choropleth_gdf = mcpp_choropleth_gdf.to_crs(epsg=4326)
else:
    mcpp_choropleth_gdf = mcpp_choropleth_gdf.set_crs(epsg=4326)

mcpp_choropleth_gdf.columns = [
    col.lower().strip()
    for col in mcpp_choropleth_gdf.columns
]

mcpp_choropleth_gdf["mcpp_neighborhood"] = (
    mcpp_choropleth_gdf["mcpp_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

mcpp_choropleth_gdf["mcpp_precinct"] = (
    mcpp_choropleth_gdf["mcpp_precinct"]
    .astype("string")
    .str.strip()
    .str.lower()
)

if "objectid" not in mcpp_choropleth_gdf.columns:
    mcpp_choropleth_gdf["objectid"] = range(1, len(mcpp_choropleth_gdf) + 1)

# ------------------------------------------------------------
# Add official MCPP labels to response_analysis
# ------------------------------------------------------------

response_mcpp = response_analysis.merge(
    event_mcpp_lookup,
    on=EVENT_ID_COLUMN,
    how="left",
)

response_mcpp["mcpp_neighborhood"] = (
    response_mcpp["mcpp_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

response_mcpp["mcpp_precinct"] = (
    response_mcpp["mcpp_precinct"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(
    "Response events with MCPP match:",
    response_mcpp["mcpp_neighborhood"].notna().sum(),
    "of",
    len(response_mcpp),
)

# ------------------------------------------------------------
# Filter to target importance bins
# ------------------------------------------------------------

TARGET_IMPORTANCE_BINS = [
    "property/nonviolent",
    "drug-related",
    "violent/person crime",
]

bin_choropleth_events = response_mcpp[
    response_mcpp["event_importance_bin"].isin(TARGET_IMPORTANCE_BINS)
].copy()

bin_choropleth_events = bin_choropleth_events[
    bin_choropleth_events["mcpp_neighborhood"].notna()
    & ~bin_choropleth_events["mcpp_neighborhood"].isin(
        ["unknown", "-", "", "nan"]
    )
].copy()

# ------------------------------------------------------------
# Overall MCPP response-time summary for selected bins
# ------------------------------------------------------------

mcpp_selected_bin_summary = (
    bin_choropleth_events
    .groupby(["mcpp_neighborhood", "mcpp_precinct"], as_index=False)
    .agg(
        target_bin_total_events=(EVENT_ID_COLUMN, "nunique"),
        target_bin_median_response_minutes=("response_time_minutes", "median"),
        target_bin_mean_response_minutes=("response_time_minutes", "mean"),
        target_bin_p90_response_minutes=(
            "response_time_minutes",
            lambda x: x.quantile(0.90),
        ),
    )
)

# ------------------------------------------------------------
# Count unique CAD events by MCPP neighborhood and bin
# ------------------------------------------------------------

mcpp_bin_counts_long = (
    bin_choropleth_events
    .groupby(["mcpp_neighborhood", "event_importance_bin"], as_index=False)
    .agg(
        bin_event_count=(EVENT_ID_COLUMN, "nunique"),
    )
)

mcpp_bin_counts_wide = (
    mcpp_bin_counts_long
    .pivot_table(
        index="mcpp_neighborhood",
        columns="event_importance_bin",
        values="bin_event_count",
        fill_value=0,
    )
    .reset_index()
)

for bin_name in TARGET_IMPORTANCE_BINS:
    if bin_name not in mcpp_bin_counts_wide.columns:
        mcpp_bin_counts_wide[bin_name] = 0

mcpp_bin_counts_wide = mcpp_bin_counts_wide.rename(
    columns={
        "property/nonviolent": "property_nonviolent_events",
        "drug-related": "drug_related_events",
        "violent/person crime": "violent_person_crime_events",
    }
)

count_cols = [
    "property_nonviolent_events",
    "drug_related_events",
    "violent_person_crime_events",
]

for col in count_cols:
    mcpp_bin_counts_wide[col] = (
        mcpp_bin_counts_wide[col]
        .fillna(0)
        .astype(int)
    )

# ------------------------------------------------------------
# Median response time by MCPP neighborhood and bin
# ------------------------------------------------------------

mcpp_bin_medians_long = (
    bin_choropleth_events
    .groupby(["mcpp_neighborhood", "event_importance_bin"], as_index=False)
    .agg(
        bin_median_response_minutes=("response_time_minutes", "median"),
    )
)

mcpp_bin_medians_wide = (
    mcpp_bin_medians_long
    .pivot_table(
        index="mcpp_neighborhood",
        columns="event_importance_bin",
        values="bin_median_response_minutes",
    )
    .reset_index()
)

for bin_name in TARGET_IMPORTANCE_BINS:
    if bin_name not in mcpp_bin_medians_wide.columns:
        mcpp_bin_medians_wide[bin_name] = pd.NA

mcpp_bin_medians_wide = mcpp_bin_medians_wide.rename(
    columns={
        "property/nonviolent": "property_nonviolent_median_response_minutes",
        "drug-related": "drug_related_median_response_minutes",
        "violent/person crime": "violent_person_crime_median_response_minutes",
    }
)

# ------------------------------------------------------------
# Combine count and response-time summaries
# ------------------------------------------------------------

mcpp_bin_summary = (
    mcpp_selected_bin_summary
    .merge(
        mcpp_bin_counts_wide,
        on="mcpp_neighborhood",
        how="left",
    )
    .merge(
        mcpp_bin_medians_wide,
        on="mcpp_neighborhood",
        how="left",
    )
)

for col in count_cols + ["target_bin_total_events"]:
    mcpp_bin_summary[col] = (
        mcpp_bin_summary[col]
        .fillna(0)
        .astype(int)
    )

# ------------------------------------------------------------
# Merge summary to official MCPP boundaries
# ------------------------------------------------------------

response_bin_choropleth = mcpp_choropleth_gdf.copy()

cols_to_drop = [
    "target_bin_total_events",
    "target_bin_median_response_minutes",
    "target_bin_mean_response_minutes",
    "target_bin_p90_response_minutes",
    "property_nonviolent_events",
    "drug_related_events",
    "violent_person_crime_events",
    "property_nonviolent_median_response_minutes",
    "drug_related_median_response_minutes",
    "violent_person_crime_median_response_minutes",
]

response_bin_choropleth = response_bin_choropleth.drop(
    columns=[col for col in cols_to_drop if col in response_bin_choropleth.columns],
    errors="ignore",
)

response_bin_choropleth = response_bin_choropleth.merge(
    mcpp_bin_summary,
    on=["mcpp_neighborhood", "mcpp_precinct"],
    how="left",
)

for col in count_cols + ["target_bin_total_events"]:
    response_bin_choropleth[col] = (
        response_bin_choropleth[col]
        .fillna(0)
        .astype(int)
    )

response_bin_choropleth["plot_feature_id"] = (
    response_bin_choropleth["objectid"]
    .astype(str)
)

response_bin_geojson = json.loads(response_bin_choropleth.to_json())

# ------------------------------------------------------------
# Choropleth shaded by median response time
# ------------------------------------------------------------

fig = px.choropleth_mapbox(
    response_bin_choropleth,
    geojson=response_bin_geojson,
    locations="plot_feature_id",
    featureidkey="properties.plot_feature_id",
    color="target_bin_median_response_minutes",
    hover_name="mcpp_neighborhood",
    custom_data=[
        "mcpp_neighborhood",
        "mcpp_precinct",
        "target_bin_median_response_minutes",
        "target_bin_mean_response_minutes",
        "target_bin_p90_response_minutes",
        "target_bin_total_events",
        "property_nonviolent_events",
        "drug_related_events",
        "violent_person_crime_events",
        "property_nonviolent_median_response_minutes",
        "drug_related_median_response_minutes",
        "violent_person_crime_median_response_minutes",
    ],
    center={
        "lat": 47.6062,
        "lon": -122.3321,
    },
    zoom=10,
    mapbox_style="carto-darkmatter",
    color_continuous_scale="Plasma",
    title=(
        "Median SPD Response Time for Selected Event Importance Bins "
        "by Official MCPP Neighborhood"
    ),
    opacity=0.75,
)

fig.update_traces(
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "MCPP precinct: %{customdata[1]}<br>"
        "<br>"
        "Selected-bin median response: %{customdata[2]:.1f} min<br>"
        "Selected-bin mean response: %{customdata[3]:.1f} min<br>"
        "Selected-bin 90th percentile: %{customdata[4]:.1f} min<br>"
        "Total selected-bin events: %{customdata[5]:,}<br>"
        "<br>"
        "<b>Counts by bin</b><br>"
        "Property/nonviolent events: %{customdata[6]:,}<br>"
        "Drug-related events: %{customdata[7]:,}<br>"
        "Violent/person crime events: %{customdata[8]:,}<br>"
        "<br>"
        "<b>Median response by bin</b><br>"
        "Property/nonviolent median: %{customdata[9]:.1f} min<br>"
        "Drug-related median: %{customdata[10]:.1f} min<br>"
        "Violent/person crime median: %{customdata[11]:.1f} min<br>"
        "<extra></extra>"
    )
)

fig.update_layout(
    margin={"r": 0, "t": 60, "l": 0, "b": 0},
    coloraxis_colorbar_title="Median response minutes",
)

fig.show()

## Response Time Patterns through 2009-2026

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

EVENT_ID_COLUMN = "cad_event_number"

PLOTLY_TEMPLATE = "plotly_dark"
PLOT_BG = "#545455"
PAPER_BG = "#111111"

RESPONSE_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "response_time"

full_history_response_path = (
    RESPONSE_OUTPUT_DIR / "spd_event_response_times_2009_2026.parquet"
)

event_response_history = pd.read_parquet(full_history_response_path)

event_response_history["queued_time"] = pd.to_datetime(
    event_response_history["queued_time"],
    errors="coerce",
)

event_response_history["first_arrival_time"] = pd.to_datetime(
    event_response_history["first_arrival_time"],
    errors="coerce",
)

event_response_history["month"] = (
    event_response_history["queued_time"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

event_response_history["year"] = event_response_history["queued_time"].dt.year
event_response_history["hour"] = event_response_history["queued_time"].dt.hour
event_response_history["day_of_week"] = event_response_history["queued_time"].dt.day_name()
event_response_history["day_of_week_num"] = event_response_history["queued_time"].dt.dayofweek

print(f"Full-history event rows: {len(event_response_history):,}")
print(event_response_history["queued_time"].min())
print(event_response_history["queued_time"].max())

In [ ]:
def assign_event_importance_bin(event_group):
    if pd.isna(event_group):
        return "unknown / unclassified"

    event_group = str(event_group).strip().lower()

    if event_group in violent_or_person_crime_groups:
        return "violent/person crime"

    if event_group in drug_related_groups:
        return "drug-related"

    if event_group in property_or_nonviolent_groups:
        return "property/nonviolent"

    if event_group in lower_public_safety_groups:
        return "lower public-safety urgency"

    return "other / unclassified"


event_response_history["event_group"] = (
    event_response_history["event_group"]
    .astype("string")
    .str.strip()
    .str.lower()
)

event_response_history["priority"] = (
    event_response_history["priority"]
    .astype("string")
    .str.strip()
    .str.lower()
)

event_response_history["dispatch_neighborhood"] = (
    event_response_history["dispatch_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

event_response_history["event_importance_bin"] = (
    event_response_history["event_group"]
    .apply(assign_event_importance_bin)
)

response_analysis = event_response_history[
    event_response_history["response_time_minutes"].notna()
    & (event_response_history["response_time_minutes"] > 0)
    & (event_response_history["response_time_minutes"] <= 24 * 60)
].copy()

response_analysis = response_analysis[
    response_analysis["queued_time"].notna()
].copy()

print(f"Events available for response-time analysis: {len(response_analysis):,}")
print(
    f"Share of full-history CAD events: "
    f"{len(response_analysis) / len(event_response_history) * 100:.2f}%"
)

response_analysis["response_time_minutes"].describe()

In [ ]:
def summarize_response_time(data, group_cols):
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    summary = (
        data
        .dropna(subset=group_cols + ["response_time_minutes"])
        .groupby(group_cols, as_index=False)
        .agg(
            unique_call_events=(EVENT_ID_COLUMN, "nunique"),
            mean_response_minutes=("response_time_minutes", "mean"),
            median_response_minutes=("response_time_minutes", "median"),
            p25_response_minutes=("response_time_minutes", lambda x: x.quantile(0.25)),
            p75_response_minutes=("response_time_minutes", lambda x: x.quantile(0.75)),
            p90_response_minutes=("response_time_minutes", lambda x: x.quantile(0.90)),
        )
    )

    return summary

Below we see the median response time from 2009-2026 for each priority, in which we can see an upward trend in both the median response time of priority 2 and priority 3 calls. The median response times for priority 3 calls was typically below 50 minutes in the period preceeding 2020, and in that same period priority 2 calls' median was typically below 20 minutes. The priority 2 median response time currently sits at 26.2 minutes, and the priority 3 median response time sits at 68.9 minutes. Priority 1 calls' median response time has also risen from under 6.5 min pre-2020 to 6.8 minutes back down from its October 2023 peak of 8.1 minutes. We can glean from this that the SPD has had trouble adjusting after 2020 and it has lead to longer response times for all priorities; nevertheless priority 1 calls still have exceptional response times relative to the other priorities.

In [ ]:
monthly_priority_response = summarize_response_time(
    response_analysis,
    ["month", "priority"],
)

monthly_priority_response = monthly_priority_response[
    monthly_priority_response["unique_call_events"] >= 30
].copy()

priority_order = (
    monthly_priority_response["priority"]
    .dropna()
    .sort_values()
    .unique()
    .tolist()
)

fig = px.line(
    monthly_priority_response,
    x="month",
    y="median_response_minutes",
    color="priority",
    markers=False,
    category_orders={
        "priority": priority_order,
    },
    title="Median SPD Response Time Over Time by Priority",
    labels={
        "month": "Month",
        "median_response_minutes": "Median response time minutes",
        "priority": "Priority",
    },
    hover_data={
        "unique_call_events": ":,",
        "median_response_minutes": ":.1f",
        "p90_response_minutes": ":.1f",
    },
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    hovermode="x unified",
    yaxis_title="Median response time minutes",
    xaxis_title="Month",
)

visible_priorities = [
    "1",
    "2",
    "3",
]

for trace in fig.data:
    if trace.name not in visible_priorities:
        trace.visible = "legendonly"


fig.show()

Below we see the median response for each event importance bin (defined previously), in which we can see longer median response times post-2020 for all bins and a huge spike in median response time during the period from April 2024 to October 2024.

In [ ]:
monthly_importance_response = summarize_response_time(
    response_analysis,
    ["month", "event_importance_bin"],
)

monthly_importance_response = monthly_importance_response[
    monthly_importance_response["unique_call_events"] >= 30
].copy()

importance_order = [
    "violent/person crime",
    "drug-related",
    "property/nonviolent",
    "lower public-safety urgency",
    "other / unclassified",
    "unknown / unclassified",
]

fig = px.line(
    monthly_importance_response,
    x="month",
    y="median_response_minutes",
    color="event_importance_bin",
    markers=False,
    category_orders={
        "event_importance_bin": importance_order,
    },
    title="Median SPD Response Time Over Time by Event Importance Bin",
    labels={
        "month": "Month",
        "median_response_minutes": "Median response time minutes",
        "event_importance_bin": "Event importance bin",
    },
    hover_data={
        "unique_call_events": ":,",
        "median_response_minutes": ":.1f",
        "p90_response_minutes": ":.1f",
    },
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    hovermode="x unified",
    yaxis_title="Median response time minutes",
    xaxis_title="Month",
)


visible_bins = [
    "violent/person crime",
    "drug-related",
    "property/nonviolent",
]

for trace in fig.data:
    if trace.name not in visible_bins:
        trace.visible = "legendonly"


fig.show()

Below examine the median response time versus the hour of the day, in which we can see that all event importance bins analyzed see spikes at 11am and 7pm. According to the [SPD website](https://seattlepolicejobs.com/faq/) the transition from the first watch to second watch occurs between 11am and 2pm, and the transition from second watch to third watch occurs between 7pm and 9pm. It seems the typical response time is spiked when watches start to switch.

In [ ]:
TARGET_IMPORTANCE_BINS = [
    "property/nonviolent",
    "drug-related",
    "violent/person crime",
]

hourly_bin_response = response_analysis[
    response_analysis["event_importance_bin"].isin(TARGET_IMPORTANCE_BINS)
].copy()

hourly_bin_response_summary = summarize_response_time(
    hourly_bin_response,
    ["hour", "event_importance_bin"],
)

hourly_bin_response_summary = (
    hourly_bin_response_summary
    .sort_values(["event_importance_bin", "hour"])
    .reset_index(drop=True)
)

fig = px.line(
    hourly_bin_response_summary,
    x="hour",
    y="median_response_minutes",
    color="event_importance_bin",
    markers=False,
    title="Median SPD Response Time by Hour of Day and Event Importance Bin",
    labels={
        "hour": "Hour of day",
        "median_response_minutes": "Median response time minutes",
        "event_importance_bin": "Event importance bin",
    },
    category_orders={
        "event_importance_bin": TARGET_IMPORTANCE_BINS,
    },
    hover_data={
        "unique_call_events": ":,",
        "median_response_minutes": ":.1f",
        "p75_response_minutes": ":.1f",
        "p90_response_minutes": ":.1f",
    },
)

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(24)),
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    yaxis_title="Median response time minutes",
    xaxis_title="Hour of day",
    legend_title_text="Event importance bin",
    hovermode="x unified",
)

fig.show()

Below we examine the median response time by the day of the week, in which we can see a small but noticeable difference between weekdays and weekends. The median response time for weekdays excluding monday is ~16.5 while the median response time for weekends is ~15.6, and interestingly monday has the slowest median responsetime at 17.1 minutes. 

In [ ]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

weekday_response_summary = summarize_response_time(
    response_analysis,
    ["day_of_week", "day_of_week_num"],
)

weekday_response_summary = (
    weekday_response_summary
    .sort_values("day_of_week_num")
    .reset_index(drop=True)
)

fig = px.bar(
    weekday_response_summary,
    x="day_of_week",
    y="median_response_minutes",
    error_y=(
        weekday_response_summary["p75_response_minutes"]
        - weekday_response_summary["median_response_minutes"]
    ),
    title="Median SPD Response Time by Day of Week",
    labels={
        "day_of_week": "Day of week",
        "median_response_minutes": "Median response time minutes",
    },
    text="median_response_minutes",
    category_orders={
        "day_of_week": day_order,
    },
    hover_data={
        "unique_call_events": ":,",
        "median_response_minutes": ":.1f",
        "p90_response_minutes": ":.1f",
    },
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    yaxis_title="Median response time minutes",
    xaxis_title="Day of week",
)

fig.show()

Below we see an interactive plot with median response time plotted over the past 17 years for each neighborhood, with the option of seeing drug-related response times or violent crime response times. The drug-related response time has been slowly climbing over the last 17 years with a noticeable uptick in 2024 for almost every neighborhood. In the violent crime response time plot we can see a similar spike in 2024 for each neighborhood, suggesting that SPD had something occur that made response times longer during that period.

In [ ]:
TARGET_IMPORTANCE_BINS = [
    "drug-related",
    "violent/person crime",
]

TOP_N_NEIGHBORHOODS = 12
MIN_MONTHLY_BIN_EVENTS = 20

response_analysis["month"] = (
    pd.to_datetime(response_analysis["queued_time"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

target_bin_response_events = response_analysis[
    response_analysis["event_importance_bin"].isin(TARGET_IMPORTANCE_BINS)
].copy()

target_bin_response_events = target_bin_response_events[
    ~target_bin_response_events["dispatch_neighborhood"].isin(
        ["unknown", "-", "", "nan"]
    )
].copy()

top_bin_neighborhoods = (
    target_bin_response_events
    .groupby("dispatch_neighborhood", as_index=False)
    .agg(
        target_bin_event_count=(EVENT_ID_COLUMN, "nunique")
    )
    .sort_values("target_bin_event_count", ascending=False)
    .head(TOP_N_NEIGHBORHOODS)
    ["dispatch_neighborhood"]
    .tolist()
)

top_bin_neighborhoods
monthly_neighborhood_bin_response = summarize_response_time(
    target_bin_response_events[
        target_bin_response_events["dispatch_neighborhood"].isin(top_bin_neighborhoods)
    ],
    ["month", "dispatch_neighborhood", "event_importance_bin"],
)


monthly_neighborhood_bin_response = monthly_neighborhood_bin_response[
    monthly_neighborhood_bin_response["unique_call_events"] >= MIN_MONTHLY_BIN_EVENTS
].copy()

monthly_neighborhood_bin_response = (
    monthly_neighborhood_bin_response
    .sort_values(["event_importance_bin", "dispatch_neighborhood", "month"])
    .reset_index(drop=True)
)

monthly_neighborhood_bin_response.head()
TARGET_IMPORTANCE_BINS = [
    "drug-related",
    "violent/person crime",
]

plot_df = monthly_neighborhood_bin_response[
    monthly_neighborhood_bin_response["event_importance_bin"].isin(TARGET_IMPORTANCE_BINS)
].copy()

plot_df = plot_df.sort_values(
    ["event_importance_bin", "dispatch_neighborhood", "month"]
).reset_index(drop=True)

fig = go.Figure()

trace_bin_labels = []

for bin_name in TARGET_IMPORTANCE_BINS:
    bin_df = plot_df[
        plot_df["event_importance_bin"] == bin_name
    ].copy()

    neighborhoods = (
        bin_df["dispatch_neighborhood"]
        .dropna()
        .sort_values()
        .unique()
        .tolist()
    )

    for neighborhood in neighborhoods:
        line_df = bin_df[
            bin_df["dispatch_neighborhood"] == neighborhood
        ].copy()

        customdata = np.stack(
            [
                line_df["dispatch_neighborhood"],
                line_df["event_importance_bin"],
                line_df["unique_call_events"],
                line_df["median_response_minutes"],
                line_df["p90_response_minutes"],
            ],
            axis=-1,
        )

        fig.add_trace(
            go.Scatter(
                x=line_df["month"],
                y=line_df["median_response_minutes"],
                mode="lines",
                name=neighborhood,
                visible=(bin_name == TARGET_IMPORTANCE_BINS[0]),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Bin: %{customdata[1]}<br>"
                    "Month: %{x|%b %Y}<br>"
                    "Events: %{customdata[2]:,}<br>"
                    "Median response: %{customdata[3]:.1f} min<br>"
                    "P90 response: %{customdata[4]:.1f} min"
                    "<extra></extra>"
                ),
            )
        )

        trace_bin_labels.append(bin_name)

buttons = []

for bin_name in TARGET_IMPORTANCE_BINS:
    visibility = [
        trace_bin == bin_name
        for trace_bin in trace_bin_labels
    ]

    buttons.append(
        dict(
            label=bin_name,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title": (
                        "Median SPD Response Time Over Time by Neighborhood "
                        f"({bin_name})"
                    )
                },
            ],
        )
    )
fig.update_layout(
    title=(
        "Median SPD Response Time Over Time by Neighborhood "
        f"({TARGET_IMPORTANCE_BINS[0]})"
    ),
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    height=720,
    margin={
        "r": 300,
        "t": 120,
        "l": 90,
        "b": 80,
    },
    xaxis_title="Month",
    yaxis_title="Median response time minutes",
    hovermode="x unified",

    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=1.02,
            y=1.02,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True,
        )
    ],

    legend=dict(
        title_text="Dispatch neighborhood",
        x=1.02,
        y=0.88,
        xanchor="left",
        yanchor="top",
    ),
)

fig.update_xaxes(
    rangeslider=dict(
        visible=True,
        thickness=0.08,
    ),
    rangeselector=dict(
        font=dict(color="black"),
        buttons=[
            dict(count=1, label="1y", step="year", stepmode="backward"),
            dict(count=3, label="3y", step="year", stepmode="backward"),
            dict(count=5, label="5y", step="year", stepmode="backward"),
            dict(count=10, label="10y", step="year", stepmode="backward"),
            dict(step="all", label="All"),
        ],
        x=0.01,
        y=1.03,
    ),
)

visible_neighborhoods = [
    "capitol hill",
]

for trace in fig.data:
    if trace.name not in visible_neighborhoods:
        trace.visible = "legendonly"


fig.show()

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import itertools

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

population_path = PROJECT_ROOT / "data" / "external" / "neighborhood_population.csv"

neighborhood_population = pd.read_csv(population_path)

neighborhood_population["dispatch_neighborhood"] = (
    neighborhood_population["dispatch_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

neighborhood_population["population"] = pd.to_numeric(
    neighborhood_population["population"],
    errors="coerce"
)

start_date = response_analysis["queued_time"].min()
end_date = response_analysis["queued_time"].max()

years_observed = (end_date - start_date).days / 365.25

print(f"Years observed: {years_observed:.2f}")
display(neighborhood_population.head())

Below we have a figure with points for each neighborhood and type of crime (neighborhoods will be plotted multiple times if multiple bins are shown) with the median response time on the y-axis, the volume of events per 1,000 residents on a logarithmic scale on the x-axis with points colored based on the event importance bins we defined earlier in the notebook (violent crimes, drug-related crimes, and property related crimes), and points sized based on the product of their response time and volume of events per 1,000 residents. The goal of this visualization is to pick out neighborhoods where there is a high volume of crimes (in any event importance bin we are interested in) and a slow median response time, the dashed lines assist us with this analysis by showing us the typical response time and typical annualized volume of events per 1,000 residents. When we consider all event importance bins we can see that property related crimes have the highest volume and slowest response times out of any of the groups; it is also clear that drug-related crime have a lower annualized volume per 1,000 residents. When we look deeper at the comparison between neighborhoods' drug-related and violent crime data we see that drug-related crimes typically have longer wait times but lower volume compared to violent crimes in the same neighborhoods, but there are a few exceptions. Sandpoint stands out as one of the neighborhoods that has higher than normal drug-related crime volume and violent crime volume while also breaking the pattern of violent crimes having faster response times than drug-related crimes. 

In [ ]:
TARGET_IMPORTANCE_BINS = [
    "property/nonviolent",
    "drug-related",
    "violent/person crime",
]

MIN_EVENTS_FOR_SCATTER = 100

scatter_response = response_analysis[
    response_analysis["event_importance_bin"].isin(TARGET_IMPORTANCE_BINS)
].copy()

scatter_response = scatter_response[
    ~scatter_response["dispatch_neighborhood"].isin(
        ["unknown", "-", "", "nan"]
    )
].copy()

volume_response_scatter = (
    scatter_response
    .groupby(["dispatch_neighborhood", "event_importance_bin"], as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        median_response_minutes=("response_time_minutes", "median"),
        mean_response_minutes=("response_time_minutes", "mean"),
        p75_response_minutes=("response_time_minutes", lambda x: x.quantile(0.75)),
        p90_response_minutes=("response_time_minutes", lambda x: x.quantile(0.90)),
    )
)

volume_response_scatter = volume_response_scatter[
    volume_response_scatter["unique_call_events"] >= MIN_EVENTS_FOR_SCATTER
].copy()

volume_response_scatter = volume_response_scatter.merge(
    neighborhood_population,
    on="dispatch_neighborhood",
    how="left"
)

volume_response_scatter = volume_response_scatter[
    volume_response_scatter["population"].notna()
    & (volume_response_scatter["population"] > 0)
].copy()

volume_response_scatter["annualized_events_per_1000"] = (
    volume_response_scatter["unique_call_events"]
    / years_observed
    / volume_response_scatter["population"]
    * 1000
)

volume_response_scatter = volume_response_scatter[
    volume_response_scatter["annualized_events_per_1000"] > 0
].copy()

volume_response_scatter.head()

def make_bin_combo_label(bin_combo):
    if len(bin_combo) == len(TARGET_IMPORTANCE_BINS):
        return "All selected bins"
    return " + ".join(bin_combo)


def add_concern_score(data):
    out = data.copy()

    out["volume_rank"] = (
        out["annualized_events_per_1000"]
        .rank(pct=True)
    )

    out["response_rank"] = (
        out["median_response_minutes"]
        .rank(pct=True)
    )

    out["concern_score"] = (
        out["volume_rank"]
        * out["response_rank"]
    )

    out["marker_size"] = (
        10 + 40 * out["concern_score"]
    )

    return out


bin_combinations = []

for r in range(1, len(TARGET_IMPORTANCE_BINS) + 1):
    for combo in itertools.combinations(TARGET_IMPORTANCE_BINS, r):
        bin_combinations.append(list(combo))

bin_combinations = [
    TARGET_IMPORTANCE_BINS
] + [
    combo for combo in bin_combinations
    if combo != TARGET_IMPORTANCE_BINS
]

fig = go.Figure()

trace_metadata = []

for combo_i, bin_combo in enumerate(bin_combinations):
    combo_label = make_bin_combo_label(bin_combo)
    combo_visible = combo_i == 0

    combo_df = volume_response_scatter[
        volume_response_scatter["event_importance_bin"].isin(bin_combo)
    ].copy()

    combo_df = add_concern_score(combo_df)

    median_event_rate = combo_df["annualized_events_per_1000"].median()
    median_of_median_response = combo_df["median_response_minutes"].median()

    for bin_name in TARGET_IMPORTANCE_BINS:
        bin_df = combo_df[
            combo_df["event_importance_bin"] == bin_name
        ].copy()

        if bin_df.empty:
            continue

        customdata = np.stack(
            [
                bin_df["dispatch_neighborhood"],
                bin_df["event_importance_bin"],
                bin_df["population"],
                bin_df["unique_call_events"],
                bin_df["annualized_events_per_1000"],
                bin_df["median_response_minutes"],
                bin_df["mean_response_minutes"],
                bin_df["p75_response_minutes"],
                bin_df["p90_response_minutes"],
                bin_df["volume_rank"],
                bin_df["response_rank"],
                bin_df["concern_score"],
            ],
            axis=-1,
        )

        fig.add_trace(
            go.Scatter(
                x=bin_df["annualized_events_per_1000"],
                y=bin_df["median_response_minutes"],
                mode="markers",
                name=bin_name,
                legendgroup=bin_name,
                showlegend=combo_i == 0,
                visible=combo_visible,
                marker=dict(
                    size=bin_df["marker_size"],
                    sizemode="diameter",
                    opacity=0.75,
                    line=dict(width=0.8, color="white"),
                ),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Bin: %{customdata[1]}<br>"
                    "Population: %{customdata[2]:,.0f}<br>"
                    "Raw unique CAD events: %{customdata[3]:,}<br>"
                    "Annualized events per 1,000 residents: %{customdata[4]:.1f}<br>"
                    "Median response: %{customdata[5]:.1f} min<br>"
                    "Mean response: %{customdata[6]:.1f} min<br>"
                    "P75 response: %{customdata[7]:.1f} min<br>"
                    "P90 response: %{customdata[8]:.1f} min<br>"
                    "<br>"
                    "Volume percentile rank: %{customdata[9]:.2f}<br>"
                    "Response percentile rank: %{customdata[10]:.2f}<br>"
                    "Concern score: %{customdata[11]:.2f}"
                    "<extra></extra>"
                ),
            )
        )

        trace_metadata.append({
            "combo_label": combo_label,
            "trace_type": "points",
        })

    fig.add_trace(
        go.Scatter(
            x=[median_event_rate, median_event_rate],
            y=[
                combo_df["median_response_minutes"].min(),
                combo_df["median_response_minutes"].max(),
            ],
            mode="lines",
            name=f"Median rate: {median_event_rate:.1f}",
            visible=combo_visible,
            showlegend=False,
            line=dict(dash="dash", width=2, color="white"),
            hovertemplate=(
                f"Median annualized event rate: {median_event_rate:.1f} per 1,000 residents"
                "<extra></extra>"
            ),
        )
    )

    trace_metadata.append({
        "combo_label": combo_label,
        "trace_type": "vertical_median",
    })

    fig.add_trace(
        go.Scatter(
            x=[
                combo_df["annualized_events_per_1000"].min(),
                combo_df["annualized_events_per_1000"].max(),
            ],
            y=[median_of_median_response, median_of_median_response],
            mode="lines",
            name=f"Median response: {median_of_median_response:.1f} min",
            visible=combo_visible,
            showlegend=False,
            line=dict(dash="dash", width=2, color="white"),
            hovertemplate=(
                f"Median response time: {median_of_median_response:.1f} min"
                "<extra></extra>"
            ),
        )
    )

    trace_metadata.append({
        "combo_label": combo_label,
        "trace_type": "horizontal_median",
    })

buttons = []

for bin_combo in bin_combinations:
    combo_label = make_bin_combo_label(bin_combo)

    visibility = [
        metadata["combo_label"] == combo_label
        for metadata in trace_metadata
    ]

    buttons.append(
        dict(
            label=combo_label,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title": (
                        "Per-Capita Call Volume vs. Median SPD Response Time "
                        f"({combo_label})"
                    )
                },
            ],
        )
    )

fig.update_layout(
    title="Per-Capita Call Volume vs. Median SPD Response Time (All selected bins)",
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    xaxis_title="Annualized unique CAD events per 1,000 residents, log scale",
    yaxis_title="Median response time minutes",
    legend_title_text="Event importance bin",
    xaxis=dict(type="log"),
    height=700,
    margin={"r": 300, "t": 120, "l": 100, "b": 80},
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=1.02,
            y=1.04,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True,
        )
    ],
    legend=dict(
        x=1.02,
        y=0.86,
        xanchor="left",
        yanchor="top",
    ),
)

fig.add_annotation(
    text="Bin selection",
    x=1.02,
    y=1.10,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="left",
    font=dict(size=13),
)

fig.show()

## Additional Figures

In [ ]:
TARGET_IMPORTANCE_BINS_NO_PROPERTY = [
    "drug-related",
    "violent/person crime",
]

MIN_EVENTS_FOR_SCATTER = 100

scatter_response_no_property = response_analysis[
    response_analysis["event_importance_bin"].isin(TARGET_IMPORTANCE_BINS_NO_PROPERTY)
].copy()

scatter_response_no_property = scatter_response_no_property[
    ~scatter_response_no_property["dispatch_neighborhood"].isin(
        ["unknown", "-", "", "nan", "null"]
    )
].copy()

combined_volume_response_scatter = (
    scatter_response_no_property
    .groupby("dispatch_neighborhood", as_index=False)
    .agg(
        selected_bin_events=(EVENT_ID_COLUMN, "nunique"),
        median_response_minutes=("response_time_minutes", "median"),
        mean_response_minutes=("response_time_minutes", "mean"),
        p75_response_minutes=("response_time_minutes", lambda x: x.quantile(0.75)),
        p90_response_minutes=("response_time_minutes", lambda x: x.quantile(0.90)),
        drug_related_events=(
            "event_importance_bin",
            lambda x: (x == "drug-related").sum()
        ),
        violent_person_crime_events=(
            "event_importance_bin",
            lambda x: (x == "violent/person crime").sum()
        ),
    )
)

combined_volume_response_scatter = combined_volume_response_scatter[
    combined_volume_response_scatter["selected_bin_events"] >= MIN_EVENTS_FOR_SCATTER
].copy()

# ---------------------------------------------
# Create a combined "concern score" for sizing
# ---------------------------------------------

combined_volume_response_scatter["volume_rank"] = (
    combined_volume_response_scatter["selected_bin_events"]
    .rank(pct=True)
)

combined_volume_response_scatter["response_rank"] = (
    combined_volume_response_scatter["median_response_minutes"]
    .rank(pct=True)
)

combined_volume_response_scatter["concern_score"] = (
    combined_volume_response_scatter["volume_rank"]
    * combined_volume_response_scatter["response_rank"]
)

combined_volume_response_scatter["marker_size"] = (
    10 + 40 * combined_volume_response_scatter["concern_score"]
)

median_selected_bin_volume = combined_volume_response_scatter["selected_bin_events"].median()
median_selected_bin_response = combined_volume_response_scatter["median_response_minutes"].median()

fig = px.scatter(
    combined_volume_response_scatter,
    x="selected_bin_events",
    y="median_response_minutes",
    size="marker_size",
    hover_name="dispatch_neighborhood",
    title=(
        "Call Volume vs. Median SPD Response Time by Neighborhood "
        "(Drug-Related and Violent/Person Crime Only)"
    ),
    labels={
        "selected_bin_events": "Selected-bin unique CAD events",
        "median_response_minutes": "Median response time minutes",
        "marker_size": "Concern score size",
    },
    hover_data={
        "selected_bin_events": ":,",
        "drug_related_events": ":,",
        "violent_person_crime_events": ":,",
        "median_response_minutes": ":.1f",
        "mean_response_minutes": ":.1f",
        "p75_response_minutes": ":.1f",
        "p90_response_minutes": ":.1f",
        "volume_rank": ":.2f",
        "response_rank": ":.2f",
        "concern_score": ":.2f",
        "marker_size": False,
    },
    log_x=True,
)

fig.update_traces(
    marker=dict(
        opacity=0.75,
        line=dict(width=0.8, color="white"),
    )
)

fig.add_vline(
    x=median_selected_bin_volume,
    line_dash="dash",
    annotation_text=f"Median volume: {median_selected_bin_volume:,.0f}",
    annotation_position="top right",
)

fig.add_hline(
    y=median_selected_bin_response,
    line_dash="dash",
    annotation_text=f"Median response: {median_selected_bin_response:.1f} min",
    annotation_position="bottom right",
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    xaxis_title="Drug-related + violent/person-crime unique CAD events, log scale",
    yaxis_title="Median response time minutes",
)

fig.show()

In [ ]:
# ------------------------------------------------------------
# Fig 1 experiment:
# Per-capita call volume vs median response time
# Points sized by high-volume + high-response concern score
# Lines connect the same neighborhood across selected bins
# Dropdown updates visible bins and median lines
# ------------------------------------------------------------

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# -----------------------------
# Load neighborhood population
# -----------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

population_path = PROJECT_ROOT / "data" / "external" / "neighborhood_population.csv"

neighborhood_population = pd.read_csv(population_path)

neighborhood_population["dispatch_neighborhood"] = (
    neighborhood_population["dispatch_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

neighborhood_population["population"] = pd.to_numeric(
    neighborhood_population["population"],
    errors="coerce"
)

# -----------------------------
# Prepare response data
# -----------------------------

TARGET_IMPORTANCE_BINS = [
    "property/nonviolent",
    "drug-related",
    "violent/person crime",
]

MIN_EVENTS_FOR_SCATTER = 100

scatter_source = response_analysis.copy()

scatter_source["queued_time"] = pd.to_datetime(
    scatter_source["queued_time"],
    errors="coerce"
)

scatter_source["dispatch_neighborhood"] = (
    scatter_source["dispatch_neighborhood"]
    .astype("string")
    .str.strip()
    .str.lower()
)

scatter_source["event_importance_bin"] = (
    scatter_source["event_importance_bin"]
    .astype("string")
    .str.strip()
    .str.lower()
)

start_date = scatter_source["queued_time"].min()
end_date = scatter_source["queued_time"].max()

years_observed = (end_date - start_date).days / 365.25

print(f"Years observed: {years_observed:.2f}")

scatter_response = scatter_source[
    scatter_source["event_importance_bin"].isin(TARGET_IMPORTANCE_BINS)
].copy()

scatter_response = scatter_response[
    ~scatter_response["dispatch_neighborhood"].isin(
        ["unknown", "-", "", "nan"]
    )
].copy()

# -----------------------------
# Aggregate to neighborhood-bin level
# -----------------------------

volume_response_scatter = (
    scatter_response
    .groupby(["dispatch_neighborhood", "event_importance_bin"], as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        median_response_minutes=("response_time_minutes", "median"),
        mean_response_minutes=("response_time_minutes", "mean"),
        p75_response_minutes=("response_time_minutes", lambda x: x.quantile(0.75)),
        p90_response_minutes=("response_time_minutes", lambda x: x.quantile(0.90)),
    )
)

volume_response_scatter = volume_response_scatter[
    volume_response_scatter["unique_call_events"] >= MIN_EVENTS_FOR_SCATTER
].copy()

volume_response_scatter = volume_response_scatter.merge(
    neighborhood_population,
    on="dispatch_neighborhood",
    how="left"
)

volume_response_scatter = volume_response_scatter[
    volume_response_scatter["population"].notna()
    & (volume_response_scatter["population"] > 0)
].copy()

volume_response_scatter["annualized_events_per_1000"] = (
    volume_response_scatter["unique_call_events"]
    / years_observed
    / volume_response_scatter["population"]
    * 1000
)

volume_response_scatter = volume_response_scatter[
    volume_response_scatter["annualized_events_per_1000"] > 0
].copy()

# -----------------------------
# Helper functions
# -----------------------------

def make_bin_combo_label(bin_combo):
    if len(bin_combo) == len(TARGET_IMPORTANCE_BINS):
        return "All selected bins"
    return " + ".join(bin_combo)


def add_concern_score(data):
    out = data.copy()

    out["volume_rank"] = (
        out["annualized_events_per_1000"]
        .rank(pct=True)
    )

    out["response_rank"] = (
        out["median_response_minutes"]
        .rank(pct=True)
    )

    out["concern_score"] = (
        out["volume_rank"]
        * out["response_rank"]
    )

    out["marker_size"] = (
        10 + 40 * out["concern_score"]
    )

    return out


# -----------------------------
# Build bin dropdown combinations
# -----------------------------

bin_combinations = []

for r in range(1, len(TARGET_IMPORTANCE_BINS) + 1):
    for combo in itertools.combinations(TARGET_IMPORTANCE_BINS, r):
        bin_combinations.append(list(combo))

bin_combinations = (
    [TARGET_IMPORTANCE_BINS]
    + [
        combo for combo in bin_combinations
        if combo != TARGET_IMPORTANCE_BINS
    ]
)

bin_color_map = {
    bin_name: px.colors.qualitative.Plotly[i]
    for i, bin_name in enumerate(TARGET_IMPORTANCE_BINS)
}

# -----------------------------
# Build figure
# -----------------------------

fig = go.Figure()
trace_metadata = []

for combo_i, bin_combo in enumerate(bin_combinations):
    combo_label = make_bin_combo_label(bin_combo)
    combo_visible = combo_i == 0

    combo_df = volume_response_scatter[
        volume_response_scatter["event_importance_bin"].isin(bin_combo)
    ].copy()

    if combo_df.empty:
        continue

    combo_df = add_concern_score(combo_df)

    median_event_rate = combo_df["annualized_events_per_1000"].median()
    median_of_median_response = combo_df["median_response_minutes"].median()

    combo_df["bin_order"] = combo_df["event_importance_bin"].apply(
        lambda x: TARGET_IMPORTANCE_BINS.index(x)
        if x in TARGET_IMPORTANCE_BINS
        else 999
    )

    # ------------------------------------------------------------
    # Experimental connector lines:
    # connect same-neighborhood points across visible bins
    # ------------------------------------------------------------

    if combo_df["event_importance_bin"].nunique() > 1:
        for neighborhood, neighborhood_df in combo_df.groupby("dispatch_neighborhood"):
            neighborhood_df = (
                neighborhood_df
                .sort_values("bin_order")
                .copy()
            )

            if len(neighborhood_df) < 2:
                continue

            fig.add_trace(
                go.Scatter(
                    x=neighborhood_df["annualized_events_per_1000"],
                    y=neighborhood_df["median_response_minutes"],
                    mode="lines",
                    name=f"{neighborhood} connector",
                    visible=combo_visible,
                    showlegend=False,
                    line=dict(
                        width=1,
                        color="rgba(255,255,255,0.35)",
                        dash="dot",
                    ),
                    hoverinfo="skip",
                )
            )

            trace_metadata.append({
                "combo_label": combo_label,
                "trace_type": "neighborhood_connector",
            })

    # -----------------------------
    # Marker traces by bin
    # -----------------------------

    for bin_name in TARGET_IMPORTANCE_BINS:
        bin_df = combo_df[
            combo_df["event_importance_bin"] == bin_name
        ].copy()

        if bin_df.empty:
            continue

        customdata = np.stack(
            [
                bin_df["dispatch_neighborhood"],
                bin_df["event_importance_bin"],
                bin_df["population"],
                bin_df["unique_call_events"],
                bin_df["annualized_events_per_1000"],
                bin_df["median_response_minutes"],
                bin_df["mean_response_minutes"],
                bin_df["p75_response_minutes"],
                bin_df["p90_response_minutes"],
                bin_df["volume_rank"],
                bin_df["response_rank"],
                bin_df["concern_score"],
            ],
            axis=-1,
        )

        fig.add_trace(
            go.Scatter(
                x=bin_df["annualized_events_per_1000"],
                y=bin_df["median_response_minutes"],
                mode="markers",
                name=bin_name,
                legendgroup=bin_name,
                showlegend=True,
                visible=combo_visible,
                marker=dict(
                    size=bin_df["marker_size"],
                    sizemode="diameter",
                    opacity=0.78,
                    color=bin_color_map[bin_name],
                    line=dict(width=0.8, color="white"),
                ),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Bin: %{customdata[1]}<br>"
                    "Population: %{customdata[2]:,.0f}<br>"
                    "Raw unique CAD events: %{customdata[3]:,}<br>"
                    "Annualized events per 1,000 residents: %{customdata[4]:.1f}<br>"
                    "Median response: %{customdata[5]:.1f} min<br>"
                    "Mean response: %{customdata[6]:.1f} min<br>"
                    "P75 response: %{customdata[7]:.1f} min<br>"
                    "P90 response: %{customdata[8]:.1f} min<br>"
                    "<br>"
                    "Volume percentile rank: %{customdata[9]:.2f}<br>"
                    "Response percentile rank: %{customdata[10]:.2f}<br>"
                    "Concern score: %{customdata[11]:.2f}"
                    "<extra></extra>"
                ),
            )
        )

        trace_metadata.append({
            "combo_label": combo_label,
            "trace_type": "points",
        })

    # -----------------------------
    # Median vertical line
    # -----------------------------

    fig.add_trace(
        go.Scatter(
            x=[median_event_rate, median_event_rate],
            y=[
                combo_df["median_response_minutes"].min(),
                combo_df["median_response_minutes"].max(),
            ],
            mode="lines",
            name=f"Median rate: {median_event_rate:.1f}",
            visible=combo_visible,
            showlegend=False,
            line=dict(dash="dash", width=2, color="white"),
            hovertemplate=(
                f"Median annualized event rate: {median_event_rate:.1f} per 1,000 residents"
                "<extra></extra>"
            ),
        )
    )

    trace_metadata.append({
        "combo_label": combo_label,
        "trace_type": "vertical_median",
    })

    # -----------------------------
    # Median horizontal line
    # -----------------------------

    fig.add_trace(
        go.Scatter(
            x=[
                combo_df["annualized_events_per_1000"].min(),
                combo_df["annualized_events_per_1000"].max(),
            ],
            y=[
                median_of_median_response,
                median_of_median_response,
            ],
            mode="lines",
            name=f"Median response: {median_of_median_response:.1f} min",
            visible=combo_visible,
            showlegend=False,
            line=dict(dash="dash", width=2, color="white"),
            hovertemplate=(
                f"Median response time: {median_of_median_response:.1f} min"
                "<extra></extra>"
            ),
        )
    )

    trace_metadata.append({
        "combo_label": combo_label,
        "trace_type": "horizontal_median",
    })

# -----------------------------
# Dropdown buttons
# -----------------------------

buttons = []

for bin_combo in bin_combinations:
    combo_label = make_bin_combo_label(bin_combo)

    visibility = [
        metadata["combo_label"] == combo_label
        for metadata in trace_metadata
    ]

    buttons.append(
        dict(
            label=combo_label,
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title": (
                        "Per-Capita Call Volume vs. Median SPD Response Time "
                        f"({combo_label})"
                    )
                },
            ],
        )
    )

# -----------------------------
# Layout
# -----------------------------

fig.update_layout(
    title="Per-Capita Call Volume vs. Median SPD Response Time (All selected bins)",
    template=PLOTLY_TEMPLATE,
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    xaxis_title="Annualized unique CAD events per 1,000 residents, log scale",
    yaxis_title="Median response time minutes",
    legend_title_text="Event importance bin",
    xaxis=dict(type="log"),
    height=720,
    margin={
        "r": 320,
        "t": 120,
        "l": 100,
        "b": 80,
    },
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=1.02,
            y=1.04,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True,
        )
    ],
    legend=dict(
        x=1.02,
        y=0.86,
        xanchor="left",
        yanchor="top",
    ),
)

fig.add_annotation(
    text="Bin selection",
    x=1.02,
    y=1.10,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="left",
    font=dict(size=13),
)

fig.show()